# 🌙 NYXARA — Google Colab me chalao (step by step)

Ye notebook **NYXARA** ko Google Colab par chalane ke liye hai. Coding aani zaroori nahi — bas har cell ko **upar se neeche, ek-ek karke** run karo.

**Cell run kaise karte hain?** Cell ke left side wale ▶️ (play) button par click karo, ya cell select karke `Shift + Enter` dabao.

| Step | Kya hoga | Kitna time |
|------|----------|-----------|
| 0 | System check (GPU mila ya nahi) | 5 sec |
| 1 | Code download (GitHub token chahiye) | 30 sec |
| 2 | Install + Colab ke liye auto-config | 2–5 min |
| 3 | *(Optional)* Google Drive — memory + model hamesha ke liye save | 1 min |
| 4 | NYXARA se baat karo! 🎉 | — |

> **⚡ Pehle ye karo:** Upar menu me **Runtime → Change runtime type → T4 GPU → Save**. NYXARA ka PRIMARY brain AiCredits hai (cloud tool) — koi bhaari model download nahi, GPU zaroori nahi. Cloud na mile to wo apni local brain se chalti hai.


## Step 0 — System check 🔍

Ye cell batayega ki GPU mila ya nahi, aur sab kuch theek hai ya nahi. Kuch install nahi karta — bas check karta hai.


In [ ]:
import os, shutil, sys

print("🔍 System check...\n")

v = sys.version_info
if (v.major, v.minor) >= (3, 11):
    print(f"✅ Python {v.major}.{v.minor} — theek hai")
else:
    print(f"❌ Python {v.major}.{v.minor} — NYXARA ko 3.11+ chahiye (Colab me normally hota hai)")

try:
    import torch
    GPU = torch.cuda.is_available()
except Exception:
    GPU = False

if GPU:
    print(f"✅ GPU mil gaya: {torch.cuda.get_device_name(0)} 🚀")
else:
    print("⚠️  GPU nahi mila — CPU par bhi chalegi, bas thodi slow rahegi.")
    print("   Fast karne ke liye: Runtime → Change runtime type → T4 GPU → Save,")
    print("   phir cells dobara upar se run karo.")

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk_gb = shutil.disk_usage("/content").free / 1e9
print(f"✅ RAM: {ram_gb:.0f} GB  |  Disk free: {disk_gb:.0f} GB")
print("\n👍 Ab Step 1 par jao.")


## Step 1 — Code download karo 📦

Repo **private** hai, isliye GitHub **Personal Access Token** chahiye. Banane ka tarika (sirf pehli baar):

1. [github.com](https://github.com) par login karo
2. Right-top profile photo → **Settings** → sabse neeche **Developer settings**
3. **Personal access tokens → Tokens (classic)** → **Generate new token (classic)**
4. Koi bhi naam do, **`repo`** wale checkbox par tick karo → **Generate token**
5. Jo `ghp_...` se shuru hone wala token dikhe, use copy kar lo

**🔑 Pro tip (recommended — token baar-baar paste nahi karna padega):**
Colab ke left sidebar me **🔑 (key) icon** par click karo → **Add new secret** → Name me `GITHUB_TOKEN` likho, Value me apna token paste karo, aur **Notebook access** toggle ON karo. Bas — ab ye cell token khud utha lega, har baar paste nahi karna padega.

Secret nahi banaya to koi baat nahi — cell khud token maangega, paste kar ke Enter dabao (token screen par dikhega nahi, ye normal hai).


In [ ]:
import os
from getpass import getpass

REPO = "nyxarajp-del/NYXARAv01"
DEST = "/content/NYXARAv01"

def _get_token():
    # Pehle Colab Secrets (🔑) me GITHUB_TOKEN dhundo, warna paste karne ko bolo
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_TOKEN")
        if t and t.strip():
            print("🔑 Token Colab Secrets se mil gaya!")
            return t.strip()
    except Exception:
        pass
    return getpass("Apna GitHub token paste karo aur Enter dabao: ").strip()

if os.path.isdir(f"{DEST}/.git"):
    # Code pehle se hai — bas latest update kheench lo
    %cd {DEST}
    token = _get_token()
    !git pull https://{token}@github.com/{REPO}.git
    print("\n✅ Code update ho gaya (latest version)!")
else:
    token = _get_token()
    !git clone https://{token}@github.com/{REPO}.git {DEST}
    assert os.path.isdir(f"{DEST}/.git"), (
        "❌ Download fail hua — upar ka error padho. Zyadatar wajah: token galat hai "
        "ya usme `repo` scope tick nahi kiya. Naya token banao aur cell dobara run karo."
    )
    %cd {DEST}
    # Token ko git config me mat chhodo (safety)
    !git remote set-url origin https://github.com/{REPO}.git
    print("\n✅ Code download ho gaya!")

del token  # token memory me mat rakho


## Step 2 — Install + auto-config ⚙️

Ye cell:
- NYXARA aur uski saari libraries install karega (**1–3 minute** — koi bhaari model download nahi (AiCredits cloud se aata hai))
- NYXARA ki settings **khud set** karega (AiCredits primary hai — ek cloud tool jise wo control karti hai; local floor bhi hamesha ready)
- End me check karega ki sab theek install hua

Beech me kuch pip warnings dikhein to **ghabrao mat, wo normal hai**. Jab tak `✅ Install complete!` na dikhe, wait karo.


In [ ]:
%cd /content/NYXARAv01

# 1) NYXARA + core libraries (reasoning, LLM, memory, security, self-training)
!pip install -q -e ".[reasoning,llm,litertlm,vector,security,observe,foundry]" httpx "pydantic>=2.6" "pydantic-settings>=2.1"

# 2) NYXARA ka PRIMARY brain ab ON-DEVICE hai: Gemma-4-E2B-it (LiteRT-LM format, ~2.4 GB).
#    Ye machine par hi chalta hai — koi API key nahi, koi network nahi, kuch bahar nahi jaata.
#    Weights agli cell utaaregi; na milein to ladder cloud rungs par khisak jaati hai. Kisi
#    bhi soorat me wo cloud par depend nahi karti.
import torch
GPU = torch.cuda.is_available()

# 3) Colab ke hisaab se NYXARA ka config (.env) likho
#    auto ladder = litertlm (primary, on-device) -> aicredits -> groq -> airouter -> self
#    -> native own-brain floor. Model file ho to har turn wahi draft karta hai; na ho to agla
#    rung sambhaal leta hai; koi cloud na mile to uski apni brain.
env_config = """# NYXARA - Colab auto-config (Step 2 ne banaya)
NYXARA_PROFILE=dev
NYXARA_LLM__PROVIDER=auto
NYXARA_LLM__AICREDITS_MODEL=moonshotai/kimi-k2-thinking
# Uska on-device primary brain (weights agli cell me aayenge).
NYXARA_LLM__LITERTLM_ENABLED=true
NYXARA_LLM__LITERTLM_AUTO_DOWNLOAD=true
# Background self-training ko Colab par calm rakho: boot par heavy architecture-search na chale
# aur foundry retraining har ~10 message ke bajaye ~50 par ho (chat responsive rahe).
NYXARA_GENESIS__RUN_ON_BOOT=false
NYXARA_AUTOFORGE__MIN_EXAMPLES=50
"""
with open(".env", "w", encoding="utf-8") as f:
    f.write(env_config)

# 4) Sanity check - import ho raha hai?
import nyxara  # noqa: F401
print(f"\n✅ Install complete!  (device: {'GPU 🚀' if GPU else 'CPU'} — primary brain: on-device Gemma-4-E2B-it)")

In [ ]:
# ---- Uska PRIMARY brain utaaro (~2.4 GB, sirf ek baar) ----------------------
# Gemma-4-E2B-it, LiteRT-LM format me — poori tarah is machine par chalta hai.
# Colab session reset hone par dobara chalana padega (disk ephemeral hai).
!python scripts/fetch_litertlm_model.py

from nyxara.mind.llm import LLM
from nyxara.kernel.config import get_settings

llm = LLM(settings=get_settings())
print("provider status:", llm.provider_status())
print("drafting on    :", llm.chosen_provider().name)


## Step 3 *(Optional, par recommended)* — Google Drive jodo 💾

Colab session band hote hi sab kuch delete ho jaata hai. Ye cell do cheezein bachata hai:

1. **NYXARA ki memory** — wo aapko yaad rakhegi, uski learning agli baar bhi bachi rahegi
2. **LLM model ka download (~1 GB)** — agli baar dobara download nahi karna padega, seedha Drive se load hoga

Sab kuch aapke Google Drive ke `NYXARA_memory` folder me jaayega. Permission popup aaye to apna Google account choose kar ke **Allow** karo.

*(Nahi chahiye to skip karo — seedha Step 4 par jao.)*


In [ ]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/NYXARA_memory"
os.makedirs(BASE, exist_ok=True)

def _link(local_path, drive_dir):
    """local_path ko Drive ke folder par point kara do (purana data move karke)."""
    os.makedirs(drive_dir, exist_ok=True)
    local_path = os.path.expanduser(local_path)
    if os.path.islink(local_path):
        os.unlink(local_path)
    elif os.path.isdir(local_path):
        for name in os.listdir(local_path):
            src, dst = os.path.join(local_path, name), os.path.join(drive_dir, name)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(local_path, ignore_errors=True)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    os.symlink(drive_dir, local_path)
    return drive_dir

# 1) NYXARA ki memory (~/.nyxara) → Drive
#    (purane notebook wale users ka data seedha NYXARA_memory me tha — use wahi rehne do)
brain_dir = BASE if os.path.exists(os.path.join(BASE, "memory.json")) else os.path.join(BASE, "brain")
print("✅ Memory ab Drive me save hogi:", _link("~/.nyxara", brain_dir))

# 2) Model cache → Drive (foundry base + koi bhi local weights dobara download nahi honge)
print("✅ Model cache bhi Drive me:", _link("~/.cache/huggingface", os.path.join(BASE, "model_cache")))


## Step 4 — NYXARA se baat karo! 🎉

Ye cell NYXARA ka console start karega.

- **Pehli baar** boot me thoda time lagega — LLM model (~1 GB) download hota hai. Agar Step 3 me Drive joda tha, to agli baar se download skip ho jayega.
- `Master>` dikhne par box me apna message type karo aur Enter dabao
- Commands ke liye `/help` type karo — jaise `/report` (status), `/wander` (usay sochne do), `/learning` (learning state), `/explain` (pichla jawab kyun aisa tha)
- **Band karne ke liye `/quit` type karo** — quit par NYXARA apni memory save kar leti hai

**Note:** Ye cell tab tak chalta rahega jab tak aap `/quit` nahi karte — ye **normal hai, error nahi**. Cell ke neeche input box me hi type karna hai.


In [ ]:
%cd /content/NYXARAv01

print("⏳ NYXARA boot ho rahi hai... (AiCredits cloud tool se — koi model download nahi)\n")

from nyxara.__main__ import main
main()

## Step 5 *(Naya)* — NYXARA ka apna 300M brain train karo 🧠

Ab tak wo cloud tools aur ek chhote apne brain par chalti thi. Ye cell uska **khud ka**
300M-parameter brain train karta hai — zero se, kisi ka pretrained weight liye bina.

**Pehle sach jaan lo, taaki waqt barbaad na ho:**

| kahan | 300M ka poora run |
|---|---|
| Colab **T4 / free GPU** | ~2-3 hafte (session 12h me katta hai — resume se hi chalega) |
| Colab **A100 (Pro+)** | ~1.5 din |
| **CPU only** | ~0.9 **saal** — code khud mana kar dega |

Isliye neeche do cells hain. **Pehla har machine par chalta hai** aur poora pipeline
sach me verify kar deta hai (chhote size par). Doosra asli 300M ke liye hai.

Jo bhi ho, `preflight` pehle bata dega — machine na keh sake to wo **shuru hi nahi karega**,
kyunki chup-chaap mahino chalna sabse bura outcome hai.


In [ ]:
%cd /content/NYXARAv01

# --- PROOF RUN: har machine par chalta hai (CPU par bhi), ~3-5 minute ---
# Ye poora pipeline chalata hai — tokenizer -> shards -> pretrain -> SFT -> eval —
# bas chhote size par. Agar ye chal gaya, to 300M ka raasta sahi hai; farak sirf compute ka hai.

!python -u scripts/train_300m.py \
    --profile nyxara-30m \
    --out /content/nyx-proof \
    --tokens 2000000 \
    --steps 300 \
    --batch-size 8 \
    --vocab-size 4096 \
    --no-compile


### Asli 300M run — sirf GPU par 🔥

Neeche wala cell asli 300M train karta hai. Do baatein:

1. **Drive zaroor jodo (Step 3)** aur `--out` ko Drive ke andar rakho. Colab session katega,
   aur `--resume` tabhi kaam karega jab checkpoints bache hon.
2. **Baar-baar chalao.** Har baar wahin se uthega jahan chhoda tha. Ye normal hai —
   300M ek baithak ka kaam nahi hai.

Achha data chahiye to `--seed-text` se apni files bhi de sakte ho (folder ya file) —
wo tokenizer aur corpus dono me sabse zyada weight paati hain.


In [ ]:
%cd /content/NYXARAv01

# --- ASLI 300M RUN (GPU) ---
# Drive par likho taaki session katne ke baad bhi bacha rahe:
OUT = "/content/drive/MyDrive/nyxara/nyx300m"   # Drive na ho to "/content/nyx300m"

# Stage 1-2: tokenizer + corpus shards (ek baar; GPU ki zarurat nahi)
!python -u scripts/train_300m.py --profile nyxara-300m --out {OUT} --stage shards --tokens 6000000000

# Stage 3: pretrain — ISE BAAR-BAAR CHALAO, har baar --resume wahin se uthayega
!python -u scripts/train_300m.py --profile nyxara-300m --out {OUT} --stage pretrain --resume \
    --steps 100000 --batch-size 8 --grad-accum 4

# Stage 4-6: SFT -> DPO -> eval (pretrain theek-thaak ho jaane ke baad)
# !python -u scripts/train_300m.py --profile nyxara-300m --out {OUT} --stage sft
# !python -u scripts/train_300m.py --profile nyxara-300m --out {OUT} --stage eval


### GPU chhota hai? Sparse profile use karo ⚡

`nyxara-moe-fast` bhi **300M parameters** ka hai, par har token par sirf **~87M** chalte hain —
yani kaafi tez, aur kam memory. Quality lagbhag ek 161M dense model jaisi hoti hai:
87M dense se kaafi behtar, dense-300M se thoda kam. Dono numbers saaf bolna zaroori hai.

```
!python -u scripts/train_300m.py --profile nyxara-moe-fast --out {OUT} --stage pretrain --resume
```


## Agli baar kya karna hai? 🔁

Colab session band hone par install delete ho jaata hai (Drive wala data bacha rehta hai). Agli baar bas:

1. Ye notebook phir kholo — Colab me **File → Open notebook → GitHub** tab, **"Include private repos"** tick karo, aur `nyxarajp-del/NYXARAv01` search karo
2. **Runtime → Change runtime type → T4 GPU** select karo
3. Step 0 se Step 4 tak cells dobara run karo (Colab Secrets me token saved hai to kuch paste bhi nahi karna padega)

Agar Step 3 me Drive joda tha, to NYXARA ki **purani memory wapas load hogi — wo aapko yaad rakhegi** 🌙 aur model bhi dobara download nahi hoga.

---

## Kuch gadbad ho to (Troubleshooting) 🔧

| Problem | Solution |
|---------|----------|
| `Your session crashed after using all available RAM` | AiCredits cloud se chalta hai, RAM shayad hi kam padegi. Runtime restart karke saare cells dobara run karo. |
| Step 1 me `Authentication failed` / `Repository not found` | Token galat ya expire hai, ya `repo` scope tick nahi kiya. Naya token banao, Colab Secret update karo, cell dobara run karo. |
| Runtime disconnect ho gaya / "Restart runtime" aa gaya | Kuch kharab nahi hua — Step 0 se saare cells dobara run kar do. |
| Step 4 wala cell rukta hi nahi | Wo **normal** hai — console chal raha hai. Band karne ke liye input box me `/quit` likho. |
| Pehli baar boot thoda slow | Cloud pehla jawab thoda dheere de sakta hai — normal hai. Uska primary model "thinking" karta hai, to pehla jawab thoda soch ke aata hai. |
| Drive me jagah nahi (`No space left`) | Google Drive me kam se kam ~3 GB free chahiye. Kuch files delete karo ya Step 3 skip kar do. |

Aur koi dikkat ho to error message copy kar ke ChatGPT/Claude se pooch lo, ya repo me issue khol do. 🌙
